### mmvec Heatmap with MASST, directionality, and ClassyFire superclass annotations

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
from os.path import join

In [4]:
# Data 
cond = pd.read_csv(
    "../data/mmvec/annotated_conditionals_SZ_MC_PD_0507202_topASV40Met_PLSDA_06022025.tsv",
    sep='\t', index_col=0
)

original_ids = pd.read_csv(
    "../data/mmvec/subset_conditionals_topASV40Met_PLSDA_06022025.tsv",
    sep='\t', index_col=0
).index.astype(str)

info_df = pd.read_csv(
    os.path.expanduser(
        "../data/mmvec/info_feature_complete_Britta_Shiseido_FBMN_2_attempt3quant_SZ_MC_PD_05072025.csv"
    )
)

VIPs_extreme_Load = pd.read_csv(
    "../data/mmvec/VIPs_extreme_Load_data_SZ_MC_PD_05072025_cytoscapeannotation.csv"
)

masst = pd.read_csv(
    "../data/mmvec/masst_screening_results.csv"
)

# Assign colors
superclass_colors = {
    "Alkaloids and derivatives": "#e41a1c", "Benzenoids": "#377eb8",
    "Hydrocarbons": "#4daf4a", "Lignans, neolignans and related compounds": "#984ea3",
    "Lipids and lipid-like molecules": "#ff7f00", "Nucleosides, nucleotides, and analogues": "#a65628",
    "Organic 1,3-dipolar compounds": "#f781bf", "Organic acids and derivatives": "#2ca25e",
    "Organic nitrogen compounds": "#66c2a5", "Organic oxygen compounds": "#8da0cb",
    "Organic salts": "#fc8d62", "Organoheterocyclic compounds": "#e78ac3",
    "Organosulfur compounds": "#a6d854", "Phenylpropanoids and polyketides": "#ffd92f",
    "Unknown": "#bdbdbd"
}
groupcontrib_colors = {"Low": "#4575b4", "High": "#e08214"}

masst_categories = [
    "plants", "tissue", "food", "microbes", "microbiome", "personal care product"
]
masst_colors = {
    "plants": "#458550", "tissue": "#393a92", "food": "#3391c1",
    "microbes": "#76325f", "microbiome": "#f08976", "personal care product": "#e8bad8"
}

# Feature mappings
feature_to_superclass = {
    str(int(r['Feature'])): (r['ClassyFire#superclass'] if pd.notna(r['ClassyFire#superclass']) else "Unknown")
    for _, r in info_df.iterrows() if not pd.isna(r['Feature'])
}

groupcontrib_dict = dict(zip(
    VIPs_extreme_Load['ID'].astype(str),
    VIPs_extreme_Load['GroupContrib']
))

masst_dict = (
    masst[masst['Matched_Size'] > 0]
    .groupby('Compound_ID')['Category']
    .apply(list)
    .to_dict()
)

# Color Bars
col_super = []
col_sebum = []
col_masst = {cat: [] for cat in masst_categories}

for fid in original_ids:
    fid_str = str(fid)
    col_super.append(superclass_colors.get(
        feature_to_superclass.get(fid_str, "Unknown"), "#bdbdbd"))
    col_sebum.append(groupcontrib_colors.get(
        groupcontrib_dict.get(fid_str, None), "#ffffff"))
    hits = masst_dict.get(int(fid), [])
    for cat in masst_categories:
        col_masst[cat].append(masst_colors[cat] if cat in hits else "#ffffff")

all_col_colors = [col_super, col_sebum] + [col_masst[cat] for cat in masst_categories]

# Center data
heatmap_data = cond.T
heatmap_data_centered = heatmap_data - heatmap_data.mean()

# Plot
sns.set(font_scale=0.6)
g = sns.clustermap(
    heatmap_data_centered,
    cmap="RdBu_r",
    col_colors=all_col_colors,
    method="average",
    metric="euclidean",
    figsize=(15, 12),
    dendrogram_ratio=(.1, .02),
    cbar_pos=(0.02, .8, .03, .18)
)
# Colorbar label
g.ax_cbar.set_ylabel("Log Conditional Probability", fontsize=22, rotation=90, labelpad=10)
g.ax_cbar.yaxis.set_label_position("left")
g.ax_cbar.tick_params(axis='y', which='both', labeltop=True, labelbottom=False)
g.ax_cbar.tick_params(labelsize=16)  # or try 18/20 to match the rest of the figure

# Feature name (tick) font sizes for metabolites (y) and microbes (x)
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), fontsize=22, rotation=90)
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=22)


g.fig.suptitle("Top Co-occurring Untargeted Metabolites and Microbial Genera Based on Conditional Probability", fontsize=24, y=1.04)


# Label Annotation Bars to the Right of the Color Bars
labels = ["Superclass", "Sebum group"] + [f"MASST: {cat}" for cat in masst_categories]

# Add labels using ax_col_colors
for i, label in enumerate(labels):
    g.ax_col_colors.text(
        1.01,                             # a bit to the right
        1 - (i + 0.5) / len(labels),      # evenly space vertically
        label,
        transform=g.ax_col_colors.transAxes,
        fontsize=22,
        ha='left', va='center'
    )

# Structured Legend 
handles = []
# Superclass block
handles.append(mpatches.Patch(color='none', label='ClassyFire Superclass'))
handles += [mpatches.Patch(color=c, label=l) for l, c in superclass_colors.items()]
# Sebum block
handles.append(mpatches.Patch(color='none', label='Sebum group'))
handles += [
    mpatches.Patch(color=groupcontrib_colors["Low"], label="Low Sebum (< 3.5 $\mu$g/cm²)"),
    mpatches.Patch(color=groupcontrib_colors["High"], label="High Sebum (> 16.9 $\mu$g/cm²)")
]
# MASST block
handles.append(mpatches.Patch(color='none', label='MASST categories'))
handles += [mpatches.Patch(color=c, label=l) for l, c in masst_colors.items()]

g.fig.legend(
    handles=handles,
    loc='upper left',
    bbox_to_anchor=(g.ax_heatmap.get_position().x1 + 0.3, g.ax_heatmap.get_position().y1+0.7),
    borderaxespad=0.,
    fontsize=22,
    frameon=False
)


# Save outputs 
outdir = "../figures/main"
os.makedirs(outdir, exist_ok=True)

for ext in ("pdf", "png", "svg"):
    plt.savefig(join(outdir, f"Figure6b.{ext}"), bbox_inches="tight")

plt.show()
print("Saved heatmap with 8 stacked annotation bars.")


Saved heatmap with 8 stacked annotation bars.


/var/folders/v_/f2630xts0zgcr95tld13kd4c0000gn/T/ipykernel_54892/1367113705.py:158: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
